# Naruto Hand Gesture Training (Colab)

This notebook trains a hand-gesture classifier for mudra recognition using transfer learning.
It is designed for Google Colab, but also works in local Jupyter with minimal changes.

## 1. Colab Runtime Setup (GPU + Dependencies)

Select **Runtime -> Change runtime type -> GPU** in Colab before training.

In [ ]:
# If running in Colab, install dependencies.
# In local Jupyter where these are already installed, this is safe to re-run.
%pip install -q torch torchvision scikit-learn matplotlib tqdm pillow

import torch
import torchvision

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 2. Project Paths and Configuration

Set reproducible config and file paths.

If your dataset is on Google Drive, uncomment the mount cell below and point `DATASET_ROOT` to your Drive location.

In [ ]:
# Optional: mount Google Drive in Colab
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path

SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.15
NUM_WORKERS = 2
PATIENCE = 6

# Update these paths as needed.
DATASET_ROOT = Path('/content/dataset')
OUTPUT_DIR = Path('/content/outputs')
OUTPUT_MODEL_PATH = OUTPUT_DIR / 'mudra_cnn.pt'
OUTPUT_META_PATH = OUTPUT_DIR / 'mudra_cnn.json'

TRAIN_DIR = DATASET_ROOT / 'train'
TEST_DIR = DATASET_ROOT / 'test'

print('DATASET_ROOT:', DATASET_ROOT)
print('TRAIN_DIR exists:', TRAIN_DIR.exists())
print('TEST_DIR exists:', TEST_DIR.exists())

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load and Inspect Gesture Dataset

Inspect class folders and visualize sample counts.

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError(f'Missing train/test directories under {DATASET_ROOT}')

train_classes = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
test_classes = sorted([p.name for p in TEST_DIR.iterdir() if p.is_dir()])

print('Train classes:', train_classes)
print('Test classes:', test_classes)
print('Num classes:', len(train_classes))

train_counts = {}
for cls in train_classes:
    train_counts[cls] = len([f for f in (TRAIN_DIR / cls).iterdir() if f.is_file()])

print('\nTrain sample counts:')
for cls, count in train_counts.items():
    print(f'  {cls}: {count}')

plt.figure(figsize=(12, 4))
plt.bar(train_counts.keys(), train_counts.values())
plt.xticks(rotation=45, ha='right')
plt.title('Training Samples Per Class')
plt.tight_layout()
plt.show()

# Show random sample images
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()
for ax in axes:
    cls = random.choice(train_classes)
    images = [f for f in (TRAIN_DIR / cls).iterdir() if f.is_file()]
    img_path = random.choice(images)
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    ax.set_title(cls)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Preprocess Data and Create Train/Validation Splits

Define augmentation and normalization, then create train/validation split.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(18),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

full_train_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
full_train_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_tfms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_tfms)

indices = np.arange(len(full_train_aug))
labels = np.array([label for _, label in full_train_aug.samples])

train_idx, val_idx = train_test_split(
    indices,
    test_size=VAL_RATIO,
    random_state=SEED,
    stratify=labels,
)

train_dataset = Subset(full_train_aug, train_idx)
val_dataset = Subset(full_train_eval, val_idx)

class_names = full_train_aug.classes
num_classes = len(class_names)

print('num_classes:', num_classes)
print('train size:', len(train_dataset))
print('val size:', len(val_dataset))
print('test size:', len(test_dataset))

## 5. Build DataLoaders

Create DataLoaders optimized for Colab training.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('device:', device)
print('steps per epoch:', len(train_loader))

## 6. Define Model Architecture (Transfer Learning)

Initialize a pretrained MobileNetV3 backbone and replace classifier head.

In [ ]:
weights = models.MobileNet_V3_Small_Weights.DEFAULT
model = models.mobilenet_v3_small(weights=weights)

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)
model = model.to(device)

print(model.classifier)

# Optional: freeze feature extractor for first stage training
# for param in model.features.parameters():
#     param.requires_grad = False

## 7. Configure Loss, Optimizer, and Metrics

Define optimization and metric utilities.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)


def accuracy_from_logits(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            running_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(labels.cpu().tolist())

    avg_loss = running_loss / max(total, 1)
    avg_acc = running_correct / max(total, 1)
    return avg_loss, avg_acc, all_targets, all_preds

## 8. Implement Training Loop with Validation

Train for multiple epochs with validation, scheduler, and best-checkpoint tracking.

In [ ]:
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

best_val_acc = 0.0
best_state = None
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(
            loss=f"{running_loss / max(total, 1):.4f}",
            acc=f"{running_correct / max(total, 1):.3f}",
        )

    train_loss = running_loss / max(total, 1)
    train_acc = running_correct / max(total, 1)

    val_loss, val_acc, _, _ = evaluate_epoch(model, val_loader, criterion, device)
    scheduler.step(val_acc)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch}. Best val_acc={best_val_acc:.3f}')
        break

if best_state is not None:
    model.load_state_dict(best_state)

## 9. Track Experiments and Plot Learning Curves

Visualize train/validation loss and accuracy over epochs.

In [ ]:
epochs_ran = len(history['train_loss'])
xs = list(range(1, epochs_ran + 1))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(xs, history['train_loss'], label='train_loss')
plt.plot(xs, history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Curves')

plt.subplot(1, 2, 2)
plt.plot(xs, history['train_acc'], label='train_acc')
plt.plot(xs, history['val_acc'], label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy Curves')

plt.tight_layout()
plt.show()

## 10. Evaluate on Holdout/Test Data

Compute final metrics, confusion matrix, and show misclassified examples.

In [ ]:
test_loss, test_acc, y_true, y_pred = evaluate_epoch(model, test_loader, criterion, device)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

cm = confusion_matrix(y_true, y_pred)
print('\nConfusion Matrix:')
print(cm)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Display misclassified examples
misclassified = []
for i, (img_path, target) in enumerate(test_dataset.samples):
    # Find matching prediction in evaluation order
    # Since test loader is not shuffled, y_pred aligns with dataset order.
    pred = y_pred[i]
    if pred != target:
        misclassified.append((img_path, target, pred))

print(f'Misclassified samples: {len(misclassified)}')

if misclassified:
    show_n = min(8, len(misclassified))
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()
    for ax, (img_path, target, pred) in zip(axes, misclassified[:show_n]):
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        ax.set_title(f'T:{class_names[target]} P:{class_names[pred]}', fontsize=9)
        ax.axis('off')
    for ax in axes[show_n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 11. Save and Export Trained Model

Save checkpoint payload, class mapping, config, and optional TorchScript export.

In [ ]:
payload = {
    'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
    'class_names': class_names,
    'image_size': IMAGE_SIZE,
    'val_acc': float(max(history['val_acc']) if history['val_acc'] else 0.0),
    'test_acc': float(test_acc),
}

torch.save(payload, OUTPUT_MODEL_PATH)

meta = {
    'model_path': str(OUTPUT_MODEL_PATH),
    'class_names': class_names,
    'image_size': IMAGE_SIZE,
    'seed': SEED,
    'batch_size': BATCH_SIZE,
    'epochs_requested': EPOCHS,
    'epochs_ran': len(history['train_loss']),
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'val_acc_best': payload['val_acc'],
    'test_acc': payload['test_acc'],
}
OUTPUT_META_PATH.write_text(json.dumps(meta, indent=2))

print('Saved model to:', OUTPUT_MODEL_PATH)
print('Saved metadata to:', OUTPUT_META_PATH)

# Optional TorchScript export
scripted_path = OUTPUT_DIR / 'mudra_cnn_scripted.pt'
example_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
scripted = torch.jit.trace(model.eval(), example_input)
scripted.save(str(scripted_path))
print('Saved TorchScript model to:', scripted_path)

## 12. Run Inference on New Images

Use the saved model to predict a single image or a small batch.

In [ ]:
def predict_image(image_path, model, class_names, image_size, device, topk=3):
    t = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    image = Image.open(image_path).convert('RGB')
    x = t(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
    vals, idxs = torch.topk(probs, k=min(topk, len(class_names)))
    return [(class_names[i], float(v)) for v, i in zip(vals.cpu().tolist(), idxs.cpu().tolist())]

# Example usage:
# sample_img = Path('/content/dataset/test/tiger/1.jpg')
# print(predict_image(sample_img, model, class_names, IMAGE_SIZE, device, topk=5))

In [ ]:
# Optional: Download artifacts in Google Colab
# from google.colab import files
# files.download(str(OUTPUT_MODEL_PATH))
# files.download(str(OUTPUT_META_PATH))